In [ ]:
## Setup — packages & environment

import sys
import subprocess

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = [
    'pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'scikit-learn',
    'statsmodels', 'openpyxl', 'reportlab'
]

for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except Exception:
        install(pkg)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime as dt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from statsmodels.tsa.seasonal import seasonal_decompose

RSEED = 2023
np.random.seed(RSEED)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel is installed:', ipykernel.__version__)
except Exception:
    print('ipykernel is NOT installed.')

In [ ]:
## 1. Data Loading and Temporal Preparation

# Create synthetic longitudinal engagement data
np.random.seed(RSEED)
n_students = 30
n_weeks = 16

data = []
for student_id in range(n_students):
    trend = np.linspace(5, 15, n_weeks) + np.random.normal(0, 2, n_weeks)
    for week in range(1, n_weeks + 1):
        engagement = max(0, trend[week-1] + np.random.normal(0, 1))
        data.append({
            'student_id': f'student_{student_id}',
            'week': week,
            'engagement': engagement,
            'forum_posts': max(0, np.random.poisson(2)),
            'assignments_submitted': np.random.binomial(1, 0.7)
        })

df = pd.DataFrame(data)
print('Longitudinal Engagement Data:')
print(df.head(10))
print(f'\nShape: {df.shape}')
print(f'Students: {df["student_id"].nunique()}, Weeks: {df["week"].max()}')

In [ ]:
## 2. Exploratory Time-Series Analysis

# Overall engagement trend
weekly_engagement = df.groupby('week')['engagement'].agg(['mean', 'std', 'count'])
print('Weekly Engagement Summary:')
print(weekly_engagement)

# Correlation with assignments
corr_assignment = df.groupby('week').apply(lambda x: x['engagement'].corr(x['assignments_submitted']))
print(f'\nMean correlation (engagement vs assignments submitted): {corr_assignment.mean():.3f}')

In [ ]:
## 3. Trajectory Clustering

# Extract engagement trajectories per student
trajectories = df.pivot(index='student_id', columns='week', values='engagement').fillna(0)
print('Trajectory matrix shape:', trajectories.shape)

# Standardize and cluster
scaler = StandardScaler()
traj_scaled = scaler.fit_transform(trajectories)

kmeans = KMeans(n_clusters=3, random_state=RSEED, n_init=10)
trajectory_clusters = kmeans.fit_predict(traj_scaled)

trajectories['cluster'] = trajectory_clusters
print('\nTrajectory Cluster Distribution:')
print(trajectories['cluster'].value_counts().sort_index())

In [ ]:
## 4. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. Overall engagement trend
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(weekly_engagement.index, weekly_engagement['mean'], 'o-', linewidth=2.5, markersize=8, label='Mean')
ax.fill_between(weekly_engagement.index, 
                 weekly_engagement['mean'] - weekly_engagement['std'],
                 weekly_engagement['mean'] + weekly_engagement['std'],
                 alpha=0.3, label='±1 Std Dev')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Engagement Score', fontsize=12)
ax.set_title('Mean Engagement Trend Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/01_engagement_trend.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_engagement_trend.png')

# 2. Trajectory clusters
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['red', 'blue', 'green']
for cluster in range(3):
    mask = trajectories['cluster'] == cluster
    for idx in trajectories[mask].index[:-1]:  # exclude 'cluster' column
        ax.plot(trajectories.columns[:-1], trajectories.loc[idx, trajectories.columns[:-1]], 
               alpha=0.4, color=colors[cluster], linewidth=1)
    
    # Plot cluster mean
    cluster_mean = trajectories[mask].drop(columns='cluster').mean()
    ax.plot(cluster_mean.index, cluster_mean.values, color=colors[cluster], linewidth=3, 
           marker='o', markersize=8, label=f'Cluster {cluster} (n={mask.sum()})')

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Engagement Score', fontsize=12)
ax.set_title('Student Engagement Trajectories by Cluster', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/02_trajectories.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_trajectories.png')

# 3. Cluster size distribution
fig, ax = plt.subplots(figsize=(8, 6))
cluster_counts = pd.Series(trajectory_clusters).value_counts().sort_index()
colors_bar = ['red', 'blue', 'green']
ax.bar(cluster_counts.index, cluster_counts.values, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Number of Students', fontsize=12)
ax.set_title('Distribution of Students by Trajectory Cluster', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(cluster_counts.values):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/03_cluster_sizes.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_cluster_sizes.png')

df.to_csv('longitudinal_results.csv', index=False)
print('\nSaved longitudinal analysis results')

In [ ]:
## 5. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch11_LongitudinalAnalysis_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter,
                        rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 11: Longitudinal Analysis of Engagement</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Longitudinal analysis examines how student engagement evolves over the course duration. '
    'By clustering trajectory patterns, we identify distinct engagement profiles: sustained high engagement, '
    'steady decline, and late-blooming trajectories.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Dataset & Variables</b>', styles['Heading2']))
dataset = (
    f'Engagement data from {df["student_id"].nunique()} students across {df["week"].max()} weeks. '
    'Variables include engagement score, forum posts, and assignment submission status.'
)
story.append(Paragraph(dataset, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Methods</b>', styles['Heading2']))
methods = (
    'Student engagement trajectories were extracted by reshaping data into student×week matrix. '
    'Trajectories were standardized and clustered using K-means (k=3). '
    'Cluster means provide prototypical engagement patterns.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>4. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_engagement_trend.png'):
        story.append(Image('figures/01_engagement_trend.png', width=500, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Overall engagement trend with confidence band.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/02_trajectories.png'):
        story.append(Image('figures/02_trajectories.png', width=520, height=360))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Three distinct trajectory clusters.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>5. Interpretation</b>', styles['Heading2']))
interp = (
    'Three distinct engagement patterns emerge: sustained high engagement, moderate declining engagement, '
    'and low engagement. Identifying these patterns enables targeted interventions for at-risk learners.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>6. Limitations</b>', styles['Heading2']))
lim = (
    'Engagement scores are synthetic for this demonstration. Real analyses require domain-validated metrics. '
    'Temporal autocorrelation not explicitly modeled.'
)
story.append(Paragraph(lim, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')